# 06_Unsupervised_Model_Development_LDA

In [3]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
import matplotlib.pyplot as plt
from sklearn.decomposition import TruncatedSVD
from sklearn.decomposition import LatentDirichletAllocation
import polars as pl
import joblib
from gensim.corpora import Dictionary
import duckdb


In [4]:
# Load the parquet file
INPUT_PATH = '../data/processed/clustered_narratives.parquet'
df_narratives = pd.read_parquet(
    INPUT_PATH,
    columns=[
        'Complaint ID',
        'processed_narrative'
    ]
)

In [5]:
df_narratives.head()

,Complaint ID,processed_narrative
0,3442136,claimed delivered package address never receiv...
1,3601853,got brink money pre paid card mail assuming un...
2,3300820,called creditor nelson cruz associate claimed ...
3,3739698,around opened credit card account online capit...
4,3285243,equifax sent credit card suggestion help impro...


In [6]:
print(len(df_narratives))

398004


## Train/Transform Vectorizer and LDA on 50k Sample

In [25]:
# # Company stopwords
# company_stopwords = { 'goldman', 'sachs', 'one', 'chase', 'well', 'fargo', 'capital', 'america', 'citibank', 'citi', 'jpmorgan', 'navy', 'federal', 'union'}

In [26]:
# # Train Count Vectorizer on 50k sample to save memory
# df_lda_sample = df_narratives.sample(
#     n=50_000,
#     random_state=42
# )
#
# count_vectorizer = CountVectorizer(
#     min_df=20,
#     max_df=0.90,
#     max_features=10_000,
#     # stop_words=list(company_stopwords)
# )
#
# X_count = count_vectorizer.fit_transform(
#     df_lda_sample['processed_narrative']
# )
#
# print(X_count.shape)

(50000, 6299)


In [7]:
# If re-running notebook, load the trained Vectorizer
count_vectorizer = joblib.load('count_vectorizer.pkl')

In [8]:
# Transform without re-training the Vectorizer
df_lda_sample = df_narratives.sample(
    n=50_000,
    random_state=42
)

X_count = count_vectorizer.transform(
    df_lda_sample['processed_narrative']
)

print(X_count.shape)

(50000, 6299)


In [10]:
# # Train LDA on sample
# # n_components = 20 was identified by the grid search later in the notebook.
# lda = LatentDirichletAllocation(
#     n_components=20,
#     learning_method='online',
#     random_state=42,
#     n_jobs=-1,
#     batch_size=2048
# )
#
# lda.fit(X_count)

In [11]:
# If re-running notebook, load the trained LDA model
lda = joblib.load('lda_model.pkl')

## Examine Topic Terms

In [12]:
# Print top terms in the LDA topics
feature_names = np.array(count_vectorizer.get_feature_names_out())

for topic_idx, topic in enumerate(lda.components_):
    top_indices = topic.argsort()[::-1][:15]
    top_terms = feature_names[top_indices]

    print(f'\nTopic {topic_idx}')
    print(', '.join(top_terms))


Topic 0
financial, issue, complaint, account, request, despite, matter, regarding, customer, resolution, action, provide, without, service, concern

Topic 1
payment, late, account, due, pay, fee, paid, made, balance, month, statement, amount, credit, time, bill

Topic 2
told, said, call, back, get, would, called, time, account, money, could, phone, asked, day, number

Topic 3
mortgage, loan, property, escrow, tax, servicing, foreclosure, servicer, shellpoint, borrower, document, notice, court, request, payment

Topic 4
account, bank, closed, checking, transfer, fund, would, citibank, saving, day, told, close, money, access, received

Topic 5
charge, card, letter, received, credit, sent, fraud, discover, fraudulent, called, email, made, number, call, statement

Topic 6
credit, card, account, limit, score, report, closed, synchrony, bank, balance, year, never, application, reason, apple

Topic 7
reporting, credit, information, account, report, consumer, debt, act, agency, inaccurate, re

In [13]:
# Add dominant topic to the sample dataframe.
# Add topic_probability which ONLY contains the probability of the dominant topic
topic_probs = lda.transform(X_count)

df_lda_sample['dominant_topic'] = topic_probs.argmax(axis=1)
df_lda_sample['topic_probability'] = topic_probs.max(axis=1)

df_lda_sample['dominant_topic'].value_counts().sort_index()

dominant_topic
0     2486
1     3994
2     9063
3      747
4     4502
5     2289
6     2310
7     1350
8     1992
9     2388
10    2406
11    2889
12    2417
13    2474
14    1111
15    1510
16      78
17     819
18    1799
19    3376
Name: count, dtype: int64

In [14]:
df_lda_sample.head()

,Complaint ID,processed_narrative,dominant_topic,topic_probability
102450,7148773,complain open bank texas day ago received regu...,9,0.300276
78236,5432970,well fargo credit card file police report repo...,11,0.495463
384187,3850335,unresolved issue citibank lasted two month sti...,4,0.435029
272901,12076787,checked account today noticed encoding error c...,19,0.418023
382568,7974309,account saving checking account closed know st...,4,0.601591


## Inspect Sample Narratives

In [15]:
# Load in cleaned narratives for the Complaint ID's in the LDA 50k sample
sample_ids = df_lda_sample['Complaint ID'].to_list()

df_readable = (
    pl.scan_parquet(INPUT_PATH)
    .filter(pl.col('Complaint ID').is_in(sample_ids))
    .select([
        'Complaint ID',
        'cleaned_consumer_narrative'
    ])
    .collect()
    .to_pandas()
)

df_lda_sample = df_lda_sample.merge(
    df_readable,
    on='Complaint ID',
    how='left'
)

In [16]:
print(df_lda_sample.columns.tolist())

['Complaint ID', 'processed_narrative', 'dominant_topic', 'topic_probability', 'cleaned_consumer_narrative']


In [17]:
# Print and inspect example narratives in each topic
for topic in sorted(df_lda_sample['dominant_topic'].unique()):
    print(f'\n===== Topic {topic} =====')

    samples = (
        df_lda_sample[df_lda_sample['dominant_topic'] == topic]
        .sample(n=3, random_state=42)
    )

    for text in samples['cleaned_consumer_narrative']:
        print('\n', text[:500])


===== Topic 0 =====

 In early REDACTED / REDACTED /year>, American Express suddenly closed all of my personal and business credit cards, as well as my charge cards, after I was told that my financial review had not been approved. I had been an American Express customer in good standing for several years, with no missed or late payments, and I always paid my balances in full or on time. During the financial review, I was asked to provide documents verifying my income and financial capability. I submitted all requeste

 On REDACTED / REDACTED /year> my Bank of America account was suddenly blocked without any prior warning or notification.What happened before the account closure : Shortly before the account was blocked, I received a REDACTED transfer for {$660.00} from a friend named REDACTED REDACTED . This money was payment for a personal debt he owed me - a legitimate transaction between friends.The only other unusual activity on my account was that I had been using it in two locatio

## Get Topic Coherence Score on Current Model

In [35]:
# tokenized_docs = (
#     df_lda_sample['processed_narrative']
#     .str.split()
#     .tolist()
# )
#
# dictionary = Dictionary(tokenized_docs)

In [36]:
# topics = []
#
# for topic in lda.components_:
#
#     top_indices = topic.argsort()[::-1][:15]
#
#     topics.append(
#         [feature_names[i] for i in top_indices]
#     )

In [37]:
# from gensim.models.coherencemodel import CoherenceModel
#
# coherence_model = CoherenceModel(
#     topics=topics,
#     texts=tokenized_docs,
#     dictionary=dictionary,
#     coherence='c_v'
# )
#
# coherence_score = coherence_model.get_coherence()
#
# print(f'Coherence Score: {coherence_score:.4f}')

Coherence Score: 0.4901


## Search for Best Coherence Score

This section searches across different values of `n_topics` using the existing 50k LDA sample, `count_vectorizer`, `X_count`, and `feature_names`.

The search evaluates each model with Gensim's `c_v` coherence score. Higher coherence is better.


In [18]:
# # Helper function to extract top words from each fitted LDA topic
# def get_lda_topics(
#     lda_model,
#     feature_names,
#     n_top_words=15
# ):
#     topics = []
#
#     for topic in lda_model.components_:
#         top_indices = topic.argsort()[::-1][:n_top_words]
#         top_words = [feature_names[i] for i in top_indices]
#         topics.append(top_words)
#
#     return topics


In [19]:
# from gensim.models.coherencemodel import CoherenceModel
#
# # Use the existing LDA sample for coherence scoring
# tokenized_docs = (
#     df_lda_sample['processed_narrative']
#     .fillna('')
#     .str.split()
#     .tolist()
# )
#
# dictionary = Dictionary(tokenized_docs)
#
# lda_coherence_results = []
#
# for n_topics in [5, 10, 15, 20, 25, 30]:
#
#     lda_search = LatentDirichletAllocation(
#         n_components=n_topics,
#         learning_method='online',
#         random_state=42,
#         n_jobs=-1,
#         batch_size=2048
#     )
#
#     lda_search.fit(X_count)
#
#     topics = get_lda_topics(
#         lda_model=lda_search,
#         feature_names=feature_names,
#         n_top_words=15
#     )
#
#     coherence_model = CoherenceModel(
#         topics=topics,
#         texts=tokenized_docs,
#         dictionary=dictionary,
#         coherence='c_v'
#     )
#
#     coherence_score = coherence_model.get_coherence()
#
#     lda_coherence_results.append({
#         'n_topics': n_topics,
#         'coherence_c_v': coherence_score,
#         'perplexity': lda_search.perplexity(X_count)
#     })
#
#     print(
#         f'n_topics={n_topics}, '
#         f'coherence_c_v={coherence_score:.4f}'
#     )
#
# lda_coherence_results_df = pd.DataFrame(lda_coherence_results)
#
# lda_coherence_results_df = lda_coherence_results_df.sort_values(
#     by='coherence_c_v',
#     ascending=False
# )
#
# lda_coherence_results_df


n_topics=5, coherence_c_v=0.4553
n_topics=10, coherence_c_v=0.4818
n_topics=15, coherence_c_v=0.4882
n_topics=20, coherence_c_v=0.4901
n_topics=25, coherence_c_v=0.4795
n_topics=30, coherence_c_v=0.4499


,n_topics,coherence_c_v,perplexity
3,20,0.490149,869.956905
2,15,0.488172,881.202929
1,10,0.481773,904.203078
4,25,0.479543,864.910800
0,5,0.455305,957.083565
5,30,0.449877,867.629379


In [38]:
# # Optional: save the topic coherence search results
# COHERENCE_RESULTS_PATH = '../data/processed/lda_topic_coherence_search.csv'
#
# lda_coherence_results_df.to_csv(
#     COHERENCE_RESULTS_PATH,
#     index=False
# )
#
# print(f'LDA coherence search results saved to: {COHERENCE_RESULTS_PATH}')


## Add Topic Probabilities for the 50k Sample as Features

In [18]:
# Create a dataframe with the probabilities of each topic
topic_features = pd.DataFrame(
    topic_probs,
    columns=[
        f'topic_{i}_prob'
        for i in range(topic_probs.shape[1])
    ]
)

# Horizontal concatenation - brings the topic probability columns into the original sample dataframe
df_lda_sample = pd.concat(
    [df_lda_sample.reset_index(drop=True),
     topic_features.reset_index(drop=True)],
    axis=1
)

df_lda_sample.head()

,Complaint ID,processed_narrative,dominant_topic,topic_probability,cleaned_consumer_narrative,topic_0_prob,topic_1_prob,topic_2_prob,topic_3_prob,topic_4_prob,...,topic_10_prob,topic_11_prob,topic_12_prob,topic_13_prob,topic_14_prob,topic_15_prob,topic_16_prob,topic_17_prob,topic_18_prob,topic_19_prob
0,7148773,complain open bank texas day ago received regu...,9,0.300276,I have a complain open No REDACTED against Ban...,0.043817,0.000485,0.296702,0.000485,0.000485,...,0.000485,0.000485,0.000485,0.152601,0.000485,0.000485,0.000485,0.000485,0.000485,0.000485
1,5432970,well fargo credit card file police report repo...,11,0.495463,Wells Fargo credit card {$3500.00} REDACTED / ...,0.002941,0.002941,0.002941,0.002941,0.002941,...,0.002941,0.495463,0.002941,0.002941,0.260341,0.002941,0.002941,0.002941,0.002941,0.002941
2,3850335,unresolved issue citibank lasted two month sti...,4,0.435029,My unresolved issues with CitiBank have now la...,0.000099,0.000099,0.238830,0.000099,0.435029,...,0.030489,0.000099,0.071104,0.000099,0.000099,0.171067,0.000099,0.000099,0.000099,0.000099
3,12076787,checked account today noticed encoding error c...,19,0.418023,I checked my account today and noticed that th...,0.000446,0.000446,0.166560,0.000446,0.000446,...,0.000446,0.000446,0.151287,0.000446,0.164227,0.000446,0.000446,0.000446,0.044577,0.418023
4,7974309,account saving checking account closed know st...,4,0.601591,My account savings and checking account was cl...,0.003125,0.003125,0.003125,0.003125,0.601591,...,0.003125,0.342159,0.003125,0.003125,0.003125,0.003125,0.003125,0.003125,0.003125,0.003125


In [20]:
# Cut down the sample dataframe to only the LDA features,
# e.g., removing narrative columns.
lda_features_sample = df_lda_sample[
    ['Complaint ID', 'dominant_topic']
    + [
        col for col in df_lda_sample.columns
        if col.startswith('topic_')
    ]
].copy()

## Apply LDA to the Remaining Records

In [21]:
# Identify remaining records outside of initial LDA sample
remaining_mask = ~df_narratives['Complaint ID'].isin(
    df_lda_sample['Complaint ID']
)

df_lda_remaining = df_narratives.loc[
    remaining_mask,
    ['Complaint ID', 'processed_narrative']
].copy()

print(f'Remaining records: {len(df_lda_remaining):,}')

Remaining records: 348,004


In [22]:
# Apply LDA to the remaining records without re-training
batch_size = 50_000
feature_batches = []

for start in range(0, len(df_lda_remaining), batch_size):
    end = min(start + batch_size, len(df_lda_remaining))

    print(f'Processing rows {start:,} to {end:,}...')

    batch = df_lda_remaining.iloc[start:end]

    X_batch = count_vectorizer.transform(
        batch['processed_narrative']
    )

    topic_probs = lda.transform(X_batch)

    topic_features = pd.DataFrame(
        topic_probs,
        columns=[
            f'topic_{i}_prob'
            for i in range(topic_probs.shape[1])
        ]
    )

    topic_features['Complaint ID'] = batch['Complaint ID'].values
    topic_features['dominant_topic'] = topic_probs.argmax(axis=1)

    feature_batches.append(topic_features)

Processing rows 0 to 50,000...
Processing rows 50,000 to 100,000...
Processing rows 100,000 to 150,000...
Processing rows 150,000 to 200,000...
Processing rows 200,000 to 250,000...
Processing rows 250,000 to 300,000...
Processing rows 300,000 to 348,004...


In [42]:
# # Write fitted Count Vectorizer and LDA to file
# joblib.dump(
#     count_vectorizer,
#     'count_vectorizer.pkl'
# )
#
# joblib.dump(
#     lda,
#     'lda_model.pkl'
# )

['lda_model.pkl']

## Combine the Sample Dataframe with the Remaining Dataframe

In [23]:
# Stack the dataframes in the feature_batches list vertically into one dataframe.

# Feature_batches contains the dominant topics and topic probabilities for the
# remaining LDA records outside the initial 50k sample.
lda_features_remaining = pd.concat(
    feature_batches,
    ignore_index=True
)

# Specify only the LDA feature columns
cols = (
    ['Complaint ID', 'dominant_topic']
    + [
        col for col in lda_features_remaining.columns
        if col.startswith('topic_')
    ]
)

# Force both dataframes to have the same columns
lda_features_remaining = lda_features_remaining[cols]
lda_features_sample = lda_features_sample[cols]

# Concatenate the 50k LDA sample with the remaining records
lda_features_all = pd.concat(
    [lda_features_sample, lda_features_remaining],
    ignore_index=True
)

print(f'Sample rows: {len(lda_features_sample):,}')
print(f'Remaining rows: {len(lda_features_remaining):,}')
print(f'Total LDA feature rows: {len(lda_features_all):,}')

Sample rows: 50,000
Remaining rows: 348,004
Total LDA feature rows: 398,004


## Export LDA Features

In [24]:
lda_features_all.head()

,Complaint ID,dominant_topic,topic_0_prob,topic_1_prob,topic_2_prob,topic_3_prob,topic_4_prob,topic_5_prob,topic_6_prob,topic_7_prob,...,topic_10_prob,topic_11_prob,topic_12_prob,topic_13_prob,topic_14_prob,topic_15_prob,topic_16_prob,topic_17_prob,topic_18_prob,topic_19_prob
0,7148773,9,0.043817,0.000485,0.296702,0.000485,0.000485,0.106598,0.000485,0.093212,...,0.000485,0.000485,0.000485,0.152601,0.000485,0.000485,0.000485,0.000485,0.000485,0.000485
1,5432970,11,0.002941,0.002941,0.002941,0.002941,0.002941,0.002941,0.194197,0.002941,...,0.002941,0.495463,0.002941,0.002941,0.260341,0.002941,0.002941,0.002941,0.002941,0.002941
2,3850335,4,0.000099,0.000099,0.238830,0.000099,0.435029,0.000099,0.017631,0.009844,...,0.030489,0.000099,0.071104,0.000099,0.000099,0.171067,0.000099,0.000099,0.000099,0.000099
3,12076787,19,0.000446,0.000446,0.166560,0.000446,0.000446,0.000446,0.049077,0.000446,...,0.000446,0.000446,0.151287,0.000446,0.164227,0.000446,0.000446,0.000446,0.044577,0.418023
4,7974309,4,0.003125,0.003125,0.003125,0.003125,0.601591,0.003125,0.003125,0.003125,...,0.003125,0.342159,0.003125,0.003125,0.003125,0.003125,0.003125,0.003125,0.003125,0.003125


In [25]:
lda_features_all.info()

<class 'pandas.DataFrame'>
RangeIndex: 398004 entries, 0 to 398003
Data columns (total 22 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   Complaint ID    398004 non-null  int64  
 1   dominant_topic  398004 non-null  int64  
 2   topic_0_prob    398004 non-null  float64
 3   topic_1_prob    398004 non-null  float64
 4   topic_2_prob    398004 non-null  float64
 5   topic_3_prob    398004 non-null  float64
 6   topic_4_prob    398004 non-null  float64
 7   topic_5_prob    398004 non-null  float64
 8   topic_6_prob    398004 non-null  float64
 9   topic_7_prob    398004 non-null  float64
 10  topic_8_prob    398004 non-null  float64
 11  topic_9_prob    398004 non-null  float64
 12  topic_10_prob   398004 non-null  float64
 13  topic_11_prob   398004 non-null  float64
 14  topic_12_prob   398004 non-null  float64
 15  topic_13_prob   398004 non-null  float64
 16  topic_14_prob   398004 non-null  float64
 17  topic_15_prob   39800

In [26]:
# Export parquet ONLY containing the LDA feature columns.
# To be joined back to the full dataframe and cluster features in the next notebook.
OUTPUT_PATH = '../data/processed/lda_features.parquet'

lda_features_all.to_parquet(
    OUTPUT_PATH,
    index=False
)

print(lda_features_all.shape)

(398004, 22)
